In [5]:
import os
import math
import time
import json
import itertools
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import numpy as np
import requests

In [6]:
# =========================
# CONFIG
# =========================
GRAPHOPPER_BASE_URL = "http://localhost:8989/route"   # endpoint /route del tuo server locale
PROFILE = "car"                                       # profilo: car | bike | foot | …
CONCURRENCY = min(32, os.cpu_count() * 4)             # numero di richieste parallele
TIMEOUT = 30                                          # timeout HTTP in secondi
MAX_RETRIES = 5                                       # retry con backoff
BACKOFF_BASE = 1.5                                    # moltiplicatore backoff
INPUT_CSV = "municipalities_trentino.csv"             # percorso al CSV
OUT_DISTANCE = "distance_km_matrix.csv"
OUT_TIME = "time_hhmm_matrix.csv"
DATADIR = "data"
# Se il tuo GraphHopper usa 'vehicle' invece di 'profile', imposta USE_VEHICLE=True
USE_VEHICLE = False  # auto: False (versioni recenti) | True (versioni più

In [7]:
# =========================
# UTILS
# =========================
def ms_to_hhmm(ms: int) -> str:
    """Converte millisecondi in stringa hh:mm (arrotondando per difetto i minuti)."""
    total_minutes = ms // 60000
    h = total_minutes // 60
    m = total_minutes % 60
    return f"{h}:{m:02d}"

def gh_route(lat1, lon1, lat2, lon2):
    """Chiama /route e ritorna (distance_m, time_ms). Retry con backoff su errori transitori."""
    params = {
        "point": [f"{lat1},{lon1}", f"{lat2},{lon2}"],
        "points_encoded": "false",
        "calc_points": "false",  # non scarichiamo la geometria
        "locale": "it"
    }
    if USE_VEHICLE:
        params["vehicle"] = PROFILE
    else:
        params["profile"] = PROFILE

    delay = 0.5
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = requests.get(GRAPHOPPER_BASE_URL, params=params, timeout=TIMEOUT)
            # Gestione rate limit / errori temporanei
            if r.status_code in (429, 502, 503, 504):
                raise requests.HTTPError(f"Temporary HTTP {r.status_code}")
            r.raise_for_status()
            data = r.json()

            # Struttura attesa: {"paths":[{"distance": metri, "time": ms, ...}]}
            paths = data.get("paths", [])
            if not paths:
                raise ValueError(f"Nessun path in risposta: {json.dumps(data)[:400]}")

            dist_m = paths[0].get("distance", None)
            time_ms = paths[0].get("time", None)
            if dist_m is None or time_ms is None:
                raise ValueError(f"Campi distance/time mancanti: {json.dumps(paths[0])[:400]}")

            return dist_m, time_ms

        except Exception as e:
            if attempt == MAX_RETRIES:
                raise
            time.sleep(delay)
            delay *= BACKOFF_BASE

def build_matrices(df):
    """Costruisce le matrici distanza (km) e tempo (hh:mm) per tutti i comuni."""
    # Indici e lookup coordinate
    ids = df["id"].tolist()
    names = df["comune"].tolist()
    coords = list(zip(df["latitude"].astype(float), df["longitude"].astype(float)))

    n = len(df)
    dist_km = np.zeros((n, n), dtype=float)
    time_ms = np.zeros((n, n), dtype=np.int64)

    # Diagonale a zero
    np.fill_diagonal(dist_km, 0.0)
    np.fill_diagonal(time_ms, 0)

    # Prepara tutte le coppie (upper triangle, esclusa la diagonale)
    pairs = [(i, j) for i in range(n) for j in range(i + 1, n)]

    def worker(i_j):
        i, j = i_j
        (lat1, lon1) = coords[i]
        (lat2, lon2) = coords[j]
        d_m, t_ms = gh_route(lat1, lon1, lat2, lon2)
        return i, j, d_m / 1000.0, int(t_ms)

    # Esecuzione in parallelo
    with ThreadPoolExecutor(max_workers=CONCURRENCY) as ex:
        futures = {ex.submit(worker, p): p for p in pairs}
        for fut in as_completed(futures):
            i, j = futures[fut]
            try:
                ii, jj, d_km, t_ms = fut.result()
                # Matrice simmetrica
                dist_km[ii, jj] = d_km
                dist_km[jj, ii] = d_km
                time_ms[ii, jj] = t_ms
                time_ms[jj, ii] = t_ms
            except Exception as e:
                # Segna NaN/NULL in caso di errore su quella coppia
                dist_km[i, j] = np.nan
                dist_km[j, i] = np.nan
                time_ms[i, j] = -1
                time_ms[j, i] = -1
                print(f"[WARN] Coppia ({names[i]} → {names[j]}) fallita: {e}")

    # DataFrame con etichette (righe/colonne = nomi, ma puoi usare 'id' se preferisci)
    dist_df = pd.DataFrame(dist_km, index=names, columns=names)
    time_df = pd.DataFrame([[ms_to_hhmm(v) if v >= 0 else "" for v in row] for row in time_ms],
                           index=names, columns=names)
    return dist_df, time_df


In [8]:
# ====== LOAD DATA ======
if not os.path.exists(DATADIR + os.sep + INPUT_CSV):
    raise FileNotFoundError(f"CSV non trovato: {INPUT_CSV}")

df = pd.read_csv(DATADIR + os.sep + INPUT_CSV)

# Adatta qui i nomi delle colonne se diversi
required = {"id", "comune", "latitude", "longitude"}
missing = required - set(df.columns.str.lower())
# Prova a normalizzare i nomi in minuscolo per tolleranza
# (se i nomi sono con maiuscole)
if missing:
    # rinomina eventuali varianti semplici
    renmap = {}
    for col in df.columns:
        l = col.lower()
        if l in required and l != col:
            renmap[col] = l
    if renmap:
        df = df.rename(columns=renmap)

# Ricontrollo
if not required.issubset(set(df.columns)):
    raise ValueError(f"Il CSV deve contenere le colonne {required}, trovate: {list(df.columns)}")

# Ordina per name (opzionale, per avere un ordinamento ripetibile)
df = df.sort_values("comune").reset_index(drop=True)

print(f"Trovati {len(df)} comuni. Avvio calcolo matrice...")

dist_df, time_df = build_matrices(df)

Trovati 166 comuni. Avvio calcolo matrice...


In [9]:
# Salva CSV
dist_df.to_csv(DATADIR + os.sep + OUT_DISTANCE, float_format="%.3f", encoding="utf-8")
time_df.to_csv(DATADIR + os.sep + OUT_TIME, encoding="utf-8")
print(f"OK: salvato {OUT_DISTANCE} (km) e {OUT_TIME} (h:mm)")

OK: salvato distance_km_matrix.csv (km) e time_hhmm_matrix.csv (h:mm)
